In [94]:
# imports
from transformers import AutoProcessor, AutoModelForCausalLM
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn.functional as F
import os
from tqdm import tqdm
import shutil
import random
import pandas as pd
from glob import glob
import numpy as np
import seaborn as sns
from PIL import Image, ImageFilter, ImageEnhance
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import cv2
from ultralytics import YOLO
import clip
import re
import copy

# ML / evaluation (fusion classifier)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Utils
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

In [2]:
# Use GPU if available
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


In [3]:
clip_model, preprocess = clip.load("ViT-B/32", device=device)

In [4]:
yolo_model = YOLO("yolov8m.pt")

In [5]:
UCF_ROOT = Path(r"UCF Crime Dataset")
SPLIT = "Train"
NORMAL_CLASS = "NormalVideos"

split_dir = UCF_ROOT / SPLIT
print("UCF_ROOT exists:", UCF_ROOT.exists())
print("split_dir exists:", split_dir.exists())
print("classes:", [p.name for p in split_dir.iterdir() if p.is_dir()])

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

UCF_ROOT exists: True
split_dir exists: True
classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']


In [6]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
_LAST_NUM = re.compile(r"_(\d+)$")  # Abuse001_x264_120 -> 120

def frame_index(p: Path) -> int:
    m = _LAST_NUM.search(p.stem)
    return int(m.group(1)) if m else 0

In [7]:
def list_videos(split_dir: Path):
    items = []
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])

    for class_dir in class_dirs:
        video_dirs = sorted([v for v in class_dir.iterdir() if v.is_dir()])
        for video_dir in video_dirs:
            frames = [p for p in video_dir.iterdir()
                      if p.is_file() and p.suffix.lower() in IMG_EXTS]
            if not frames:
                continue
            frames = sorted(frames, key=frame_index)

            items.append({
                "class_name": class_dir.name,
                "video_id": video_dir.name,
                "video_dir": video_dir,
                "frames": frames
            })
    return items

videos = list_videos(split_dir)
print("Total videos:", len(videos))
print("Example item:", videos[0]["class_name"], videos[0]["video_id"], "frames:", len(videos[0]["frames"]))


Total videos: 1610
Example item: Abuse Abuse001_x264 frames: 273


In [8]:
classes = sorted({v["class_name"] for v in videos})
label2idx = {c:i for i,c in enumerate(classes)}
idx2label = {i:c for c,i in label2idx.items()}

print("Classes:", classes)
print("label2idx:", label2idx)


Classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
label2idx: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}


In [9]:
def uniform_sample_indices(n_frames: int, n_sample: int) -> np.ndarray:
    if n_frames <= 0:
        return np.array([], dtype=int)
    if n_frames >= n_sample:
        return np.linspace(0, n_frames - 1, num=n_sample, dtype=int)
    return np.linspace(0, n_frames - 1, num=n_sample, dtype=int) 


In [10]:
class UCFCrimeVideoDataset(Dataset):
    def __init__(self, video_items, label2idx, n_frames=32, transform=None,
                 return_binary=False, normal_class="NormalVideos"):
        self.video_items = video_items
        self.label2idx = label2idx
        self.n_frames = n_frames
        self.transform = transform
        self.return_binary = return_binary
        self.normal_class = normal_class

    def __len__(self):
        return len(self.video_items)

    def __getitem__(self, i):
        item = self.video_items[i]
        frames = item["frames"]
        idxs = uniform_sample_indices(len(frames), self.n_frames)

        imgs = []
        for k in idxs:
            img = Image.open(frames[int(k)]).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)  # tensor
            imgs.append(img)

        if self.transform is not None:
            x = torch.stack(imgs, dim=0)  # [T,3,H,W]
        else:
            x = imgs  # list of PIL

        class_name = item["class_name"]
        y_multi = self.label2idx[class_name]
        y_bin = 0 if class_name == self.normal_class else 1

        if self.return_binary:
            return x, y_bin, y_multi, item["video_id"], class_name
        return x, y_multi, item["video_id"], class_name


In [15]:
ds = UCFCrimeVideoDataset(
    videos, label2idx,
    n_frames=32,
    transform=preprocess,        
    return_binary=True,
    normal_class=NORMAL_CLASS
)

dl = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

In [55]:
@torch.no_grad()
def clip_batch_to_video_vectors(x, clip_model, device):
    """
    Convert a batch of videos (as sampled frames) into one fixed-size CLIP vector per video.

    x: [B, T, 3, 224, 224]
    returns: [B, 2D] where D is CLIP embedding dim (ViT-B/32 usually 512)
    """

    clip_model.eval()  # Set model to evaluation mode (no dropout, stable inference)

    # Unpack input dimensions
    B, T, C, H, W = x.shape

    # Move input to GPU/CPU device
    x = x.to(device)

    # Flatten videos into a single batch of frames:
    # [B, T, 3, 224, 224] -> [B*T, 3, 224, 224]
    xf = x.view(B * T, C, H, W)

    # Encode each frame with CLIP to get an embedding per frame:
    # [B*T, 3, 224, 224] -> [B*T, D]
    feats = clip_model.encode_image(xf)

    # Normalize embeddings to unit length (helps stability and similarity-based learning)
    feats = F.normalize(feats, dim=-1)

    # Reshape back to per-video sequence:
    # [B*T, D] -> [B, T, D]
    D = feats.shape[-1]
    feats = feats.view(B, T, D)

    # Pool over time to get one vector per video:
    # mean pooling captures "overall" content across frames
    v_mean = feats.mean(dim=1)            # [B, D]
    # max pooling captures the strongest signal across frames (useful if anomaly appears briefly)
    v_max  = feats.max(dim=1).values      # [B, D]

    # Concatenate mean and max to form the final video representation:
    # [B, D] + [B, D] -> [B, 2D]
    video_vecs = torch.cat([v_mean, v_max], dim=-1)

    return video_vecs

In [59]:
x, y_bin, y_multi, video_ids, class_names = next(iter(dl))  # Take one batch from DataLoader

video_vecs = clip_batch_to_video_vectors(x, clip_model, device)

print("x shape:", x.shape)                    # Expected: [B, T, 3, 224, 224]
print("video_vecs shape:", video_vecs.shape)  # Expected: [B, 2D] e.g., [2, 1024]
print("video_ids:", video_ids)                # Debug: which video folders were sampled
print("class_names:", class_names)            # Debug: their class names
print("binary:", y_bin)                       # 0=NormalVideos, 1=Anomaly
print("multi:", y_multi)                      # Multiclass label indices

x shape: torch.Size([2, 32, 3, 224, 224])
video_vecs shape: torch.Size([2, 1024])
video_ids: ('Explosion014_x264', 'Assault023_x264')
class_names: ('Explosion', 'Assault')
binary: tensor([1, 1])
multi: tensor([5, 3])


In [62]:
@torch.no_grad()
def extract_clip_embeddings_from_loader(dl, clip_model, device):
    clip_model.eval()

    X_list = []
    ybin_list = []
    ymulti_list = []
    vid_list = []
    cname_list = []

    for x, y_bin, y_multi, video_ids, class_names in dl:
        # x: [B, T, 3, 224, 224]
        video_vecs = clip_batch_to_video_vectors(x, clip_model, device)  # [B, 2D]

        # Move to CPU for storage
        X_list.append(video_vecs.cpu())
        ybin_list.append(y_bin.cpu())
        ymulti_list.append(y_multi.cpu())

        # Keep ids/names as python lists
        vid_list.extend(list(video_ids))
        cname_list.extend(list(class_names))

    X_embed = torch.cat(X_list, dim=0)
    y_bin_all = torch.cat(ybin_list, dim=0)
    y_multi_all = torch.cat(ymulti_list, dim=0)

    return X_embed, y_bin_all, y_multi_all, vid_list, cname_list

In [63]:
X_embed, y_bin_all, y_multi_all, vid_list, cname_list = extract_clip_embeddings_from_loader(dl, clip_model, device)

print("X_embed shape:", X_embed.shape)
print("y_bin shape:", y_bin_all.shape, "| positives:", int((y_bin_all == 1).sum()))
print("y_multi shape:", y_multi_all.shape)
print("Example:", vid_list[0], cname_list[0], "bin=", int(y_bin_all[0]), "multi=", int(y_multi_all[0]))

X_embed shape: torch.Size([1610, 1024])
y_bin shape: torch.Size([1610]) | positives: 810
y_multi shape: torch.Size([1610])
Example: Arson053_x264 Arson bin= 1 multi= 2


In [ ]:
# # save to disk (so you don't recompute embeddings every time)
# save_path = "ucf_clip_embeddings_train.pt"
# torch.save(
#     {
#         "X_embed": X_embed,
#         "y_bin": y_bin_all,
#         "y_multi": y_multi_all,
#         "video_ids": vid_list,
#         "class_names": cname_list,
#     },
#     save_path,
# )

# print("Saved:", save_path)

Saved: ucf_clip_embeddings_train.pt


In [65]:
data = torch.load("ucf_clip_embeddings_train.pt", map_location="cpu")

X_embed = data["X_embed"]          # [N, 1024]
y_bin_all = data["y_bin"]          # [N]
y_multi_all = data["y_multi"]      # [N]
vid_list = data["video_ids"]       # list[str]
cname_list = data["class_names"]   # list[str]

print("Loaded:")
print("X_embed:", X_embed.shape)
print("y_bin:", y_bin_all.shape, "| positives:", int((y_bin_all == 1).sum()))
print("y_multi:", y_multi_all.shape)
print("Example:", vid_list[0], cname_list[0], "bin=", int(y_bin_all[0]), "multi=", int(y_multi_all[0]))

Loaded:
X_embed: torch.Size([1610, 1024])
y_bin: torch.Size([1610]) | positives: 810
y_multi: torch.Size([1610])
Example: Arson053_x264 Arson bin= 1 multi= 2


In [66]:
# Convert to numpy for splitting
X_np = X_embed.numpy()
y_np = y_bin_all.numpy()

# Stratified split keeps the same anomaly ratio in train and val
X_train, X_val, y_train, y_val = train_test_split(
    X_np, y_np,
    test_size=0.2,
    random_state=42,
    stratify=y_np
)

print("Train:", X_train.shape, "positives:", int(y_train.sum()))
print("Val  :", X_val.shape,   "positives:", int(y_val.sum()))


Train: (1288, 1024) positives: 648
Val  : (322, 1024) positives: 162


In [68]:
# Convert to torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)  # float for BCEWithLogitsLoss

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))

train batches: 21
val batches: 2


In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, in_dim=1024, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)  # one logit
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # [B]

model = BinaryMLP(in_dim=X_train_t.shape[1], hidden_dim=256, dropout=0.3).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

print(model)

BinaryMLP(
  (net): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [ ]:
def train_model(
    model, loader, optimizer, criterion, device,
    epochs=10,
    val_loader=None,
    patience=5,
    save_best_path="best_binary_mlp.pt"
):
    train_losses = []

    best_val_loss = float("inf")
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(loader, total=len(loader), desc=f"Train Epoch {ep}/{epochs}", leave=False)

        for Xb, yb in pbar:
            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(Xb)           
            loss = criterion(logits, yb) 
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * Xb.size(0)

            # Update progress bar with current loss
            pbar.set_postfix(loss=float(loss.item()))

        avg_loss = total_loss / len(loader.dataset)
        train_losses.append(avg_loss)

        # Print epoch summary (same as you had)
        msg = f"Epoch {ep:02d}/{epochs} | train_loss={avg_loss:.4f}"

        if val_loader is not None:
            val_loss, _, _ = eval_model(model, val_loader, criterion, device)

            msg += f" | val_loss={val_loss:.4f}"

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, save_best_path)
                bad_epochs = 0
            else:
                bad_epochs += 1
                msg += f" | patience {bad_epochs}/{patience}"

            print(msg)

            if bad_epochs >= patience:
                print(f"Early stopping: no val_loss improvement for {patience} epochs.")
                break
        else:
            print(msg)

    # ---- Added: load best weights back into the model ----
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best weights | best_val_loss={best_val_loss:.4f} | saved to: {save_best_path}")

    return train_losses


@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    all_probs = []
    all_y = []

    pbar = tqdm(loader, total=len(loader), desc="Eval", leave=False)

    for Xb, yb in pbar:
        Xb = Xb.to(device)
        yb = yb.to(device)

        logits = model(Xb)
        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)

        total_loss += loss.item() * Xb.size(0)
        all_probs.append(probs.cpu())
        all_y.append(yb.cpu())

        # Update progress bar with current loss
        pbar.set_postfix(loss=float(loss.item()))

    val_loss = total_loss / len(loader.dataset)
    all_probs = torch.cat(all_probs).numpy()
    all_y = torch.cat(all_y).numpy()

    print(f"Eval | val_loss={val_loss:.4f}")
    return val_loss, all_probs, all_y


In [163]:
train_losses = train_model(
    model, train_loader, optimizer, criterion, device,
    epochs=50,
    val_loader=val_loader,
    patience=7,
    save_best_path="best_binary_mlp.pt"
)

# model is already loaded with best weights
val_loss, val_probs, val_y = eval_model(model, val_loader, criterion, device)


Eval | val_loss=0.6592
Epoch 01/50 | train_loss=0.6789 | val_loss=0.6592


Eval | val_loss=0.6156
Epoch 02/50 | train_loss=0.6407 | val_loss=0.6156


Eval | val_loss=0.5615
Epoch 03/50 | train_loss=0.5908 | val_loss=0.5615


Eval | val_loss=0.5095
Epoch 04/50 | train_loss=0.5371 | val_loss=0.5095


Eval | val_loss=0.4672
Epoch 05/50 | train_loss=0.4938 | val_loss=0.4672


...

Eval | val_loss=0.3376
Epoch 42/50 | train_loss=0.2782 | val_loss=0.3376 | patience 4/7


Eval | val_loss=0.3408
Epoch 43/50 | train_loss=0.2766 | val_loss=0.3408 | patience 5/7


Eval | val_loss=0.3400
Epoch 44/50 | train_loss=0.2711 | val_loss=0.3400 | patience 6/7


Eval | val_loss=0.3391
Epoch 45/50 | train_loss=0.2708 | val_loss=0.3391 | patience 7/7
Early stopping: no val_loss improvement for 7 epochs.
Loaded best weights | best_val_loss=0.3375 | saved to: best_binary_mlp.pt


Eval | val_loss=0.3375


In [169]:
val_pred = (val_probs >= 0.4).astype(int)

acc = accuracy_score(val_y, val_pred)
f1 = f1_score(val_y, val_pred)
cm = confusion_matrix(val_y, val_pred)

print("Accuracy:", acc)
print("F1:", f1)
print("Confusion matrix:\n", cm)
print("\nReport:\n", classification_report(val_y, val_pred, digits=4))


Accuracy: 0.8664596273291926
F1: 0.8731563421828908
Confusion matrix:
 [[131  29]
 [ 14 148]]

Report:
               precision    recall  f1-score   support

         0.0     0.9034    0.8187    0.8590       160
         1.0     0.8362    0.9136    0.8732       162

    accuracy                         0.8665       322
   macro avg     0.8698    0.8662    0.8661       322
weighted avg     0.8696    0.8665    0.8661       322



In [165]:
torch.save(model.state_dict(), "binary_clip_mlp_best.pt")
print("Saved: binary_clip_mlp_best.pt")

Saved: binary_clip_mlp_best.pt
